# 03. Gate: champion vs challenger

DM-HLN на общих фолдах, 95 % блочный бутстрап разницы RMSE и правило решения. Ряды ошибок синтетические; функции — те же, что в `tools/gate.py`.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "bi_ml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from bi_ml.eval_utils import block_bootstrap_rmse_diff, diebold_mariano, mae, rmse

In [ ]:
import numpy as np

rng = np.random.default_rng(20261110)
n = 60
err_champion = rng.normal(0, 1.0, n).tolist()
err_challenger = rng.normal(0, 0.8, n).tolist()

print(f"champion  RMSE={rmse(err_champion):.3f}  MAE={mae(err_champion):.3f}")
print(f"challenger RMSE={rmse(err_challenger):.3f}  MAE={mae(err_challenger):.3f}")

In [ ]:
stat, p_value = diebold_mariano(err_champion, err_challenger, h=1)
boot = block_bootstrap_rmse_diff(err_champion, err_challenger, block=4, n=2000, seed=20261110)

print(f"DM-HLN: statistic={stat:.3f}, p={p_value:.4f}")
print(f"ΔRMSE (champion − challenger) = {boot.rmse_diff:.3f}, "
      f"95% CI [{boot.low:.3f}, {boot.high:.3f}], p={boot.p_value:.4f}")

promote = p_value < 0.05 and boot.low > 0
print("Решение:", "promote" if promote else "iterate/reject")

## Правило gate

- Primary: DM-HLN p < 0,05 **и** 95 % CI разницы RMSE исключает 0 (challenger лучше).
- Robustness: тот же вывод на robustness-окне; минимум 12 общих фолдов в основном окне и 6 в robustness.
- MCS 90 % — только при > 2 кандидатах.
- `promote` не двигает alias и не публикует модель: для CPI — `make mlflow-promote`, для key-rate — shadow/canary.

Боевой запуск: `make ml-gate ARGS="--champion-run <uuid> --challenger-run <uuid> --model <model>"`.
